# 03 — Uplift Modeling

`02_data_prep.ipynb` produced `data/processed/{train,val,test}.parquet` and `feature_manifest.json`: stratified 60/20/20 splits on the 3-arm `segment` column, `zip_code`/`channel` one-hot encoded (`drop_first=True`), `history_segment` dropped, no scaling/imputation (deferred here, per-fold). This notebook fits uplift models on those splits.

Four things to settle before any modeling:

1. The three-arm treatment problem — scikit-uplift's meta-learners expect a binary treatment, `segment` has three levels.
2. What to model uplift on, and which meta-learner(s).
3. Which base learner(s), given the covariate structure from `01_eda.ipynb` (skew, multicollinearity, VIF).
4. How much hyperparameter search this project's scale justifies.

Scope boundary with `04`: this notebook fits models and takes an informal first look at whether rankings are plausible. The rigorous evaluation (Qini curves, uplift@k, confidence intervals) is `04_evaluation.ipynb`'s job, on `test.parquet`, untouched here.

In [1]:
%pip install torch pandas numpy scikit-learn scikit-uplift pyarrow plotly

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(project_root))

from src.data_prep import make_binary_treatment
from src.models import (
    XLearner,
    build_gradient_boosting_base_learner,
    build_gradient_boosting_effect_regressor,
    build_logistic_regression_base_learner,
    build_decision_tree_base_learner,
    build_random_forest_base_learner,
    build_mlp_base_learner,
    TARNetUpliftModel,
    compute_uplift_at_k,
    collect_uplift_predictions,
    prepare_arm_splits,
    tune_base_learner_hyperparameters,
    ensemble_uplift_predictions,
)


from sklift.metrics import qini_auc_score
from sklift.models import SoloModel, TwoModels

pd.set_option("display.max_columns", None)

with open("../data/processed/feature_manifest.json") as f:
    manifest = json.load(f)

feature_columns = manifest["feature_columns"]
treatment_col = manifest["treatment_col"]

train_df = pd.read_parquet("../data/processed/train.parquet")
val_df = pd.read_parquet("../data/processed/val.parquet")
test_df = pd.read_parquet("../data/processed/test.parquet")

print(f"train: {train_df.shape}, val: {val_df.shape}, test: {test_df.shape}")
print(f"Feature columns ({len(feature_columns)}): {feature_columns}")

train: (38400, 13), val: (12800, 13), test: (12800, 13)
Feature columns (9): ['recency', 'history', 'mens', 'womens', 'newbie', 'zip_code_Surburban', 'zip_code_Urban', 'channel_Phone', 'channel_Web']


## Step 1: The three-arm treatment problem

`segment` has three levels: `Mens E-Mail`, `Womens E-Mail`, `No E-Mail`. scikit-uplift's meta-learners (`SoloModel`, `TwoModels`, and this notebook's `XLearner`) all expect a binary treatment column. Options:

- Pool both email arms into "any email." Rejected — the two campaigns are different interventions (content, targeting rationale), not two doses of one treatment; pooling answers "what's the effect of email in general" rather than "which customers should get which email?"
- Analyze only one arm. Rejected — the experiment randomized three ways specifically to evaluate both campaigns.
- Two parallel binary analyses, each email arm vs the shared `No E-Mail` control. Used here. Each comparison is a valid two-arm RCT on its own (any two-arm subset of a fixed-probability multi-arm randomization is itself a clean two-arm RCT). Matches what `make_binary_treatment` in `src/data_prep.py` was built for — filters to one treatment arm plus control, returns a binary `treatment` column, called once per analysis.

Caveat: both analyses share the same `No E-Mail` control, so their sampling noise is correlated through it. That doesn't bias either analysis's own point estimate, but a direct claim like "uplift is higher for Mens than Womens" would need to account for that correlation — not attempted here.

`prepare_arm_splits` in `src/models.py` composes `make_binary_treatment` with feature/outcome extraction; each arm's train/val/test triples are one call below.

In [3]:
mens_train_check = make_binary_treatment(train_df, treatment_group="Mens E-Mail")
womens_train_check = make_binary_treatment(train_df, treatment_group="Womens E-Mail")

print(f"Mens E-Mail vs No E-Mail (train): {mens_train_check.shape[0]} rows")
print(mens_train_check["treatment"].value_counts())
print()
print(f"Womens E-Mail vs No E-Mail (train): {womens_train_check.shape[0]} rows")
print(womens_train_check["treatment"].value_counts())

Mens E-Mail vs No E-Mail (train): 25568 rows
treatment
0    12784
1    12784
Name: count, dtype: int64

Womens E-Mail vs No E-Mail (train): 25616 rows
treatment
1    12832
0    12784
Name: count, dtype: int64


## Step 2: What to model uplift on, and which meta-learners

**Outcome: `visit`, not `conversion`.** `01_eda.ipynb` found `visit` at about 15% and `conversion` at under 1% of customers. At this project's per-arm training size (tens of thousands of rows before the treatment/control split, roughly half that again within each arm), a sub-1% outcome leaves very few positive events for a meta-learner to learn heterogeneity from — particularly for T-Learner and X-Learner, whose later stages fit on an already-arm-split subset of an already-rare outcome. `visit` has about 15x more signal, and it isn't a disconnected proxy: it's the necessary first step in the funnel toward `conversion`, so "who should get the email to maximize the chance they engage at all" is a genuinely actionable question on its own. A lighter `conversion`-target stress test appears at the end of this notebook, on one arm and one model, specifically to make the rare-outcome problem concrete rather than just asserting it.

**Meta-learners: S-Learner, T-Learner, X-Learner.**

- **S-Learner** (`sklift.models.SoloModel`) fits one model over the pooled arm+control data, with treatment as an extra feature, and takes the difference between predicting with treatment on vs off. Cheapest of the three (one model, all the data), but has a known failure mode: a flexible base learner can fit the outcome well using the *other* covariates alone, without leaning on the treatment feature at all, which shrinks the estimated effect toward zero exactly when the treatment signal is weak relative to the rest of the covariates — plausibly the case here, given the campaign's overall lift is modest. Included as a cheap, transparent baseline, not expected to win.
- **T-Learner** (`sklift.models.TwoModels`, `method="vanilla"`) fits two fully separate models, one per arm, and takes the difference in their predictions. More expressive than S-Learner (each model can learn arm-specific relationships freely), but splits the training data in half, compounding the rare-outcome problem `visit` was chosen partly to avoid.
- **X-Learner** is hand-implemented as `XLearner` in `src/models.py`, because scikit-uplift ships `SoloModel` and `TwoModels` but no native X-Learner (see that class's docstring for the full algorithm). It's included specifically because it's designed for this project's situation: it starts from the same two arm-specific outcome models as a T-Learner, but adds a stage that imputes and directly regresses individual treatment effects, then blends the two resulting effect models via the propensity score — trusting each one more in the region of covariate space where it has more local support. Because this is a *randomized* experiment, the propensity is known by design rather than estimated: `XLearner` defaults to using the empirical treatment fraction as a constant propensity, which is simpler and lower-variance than fitting a covariate-dependent propensity model to recover, with added noise, a quantity already known exactly.

A native uplift tree/forest (which optimizes a treatment-effect-splitting criterion directly, rather than differencing two outcome models) was considered as a fourth comparison point, but scikit-uplift doesn't ship one as of the pinned version, and hand-rolling a full splitting criterion is out of scope here — the S-/T-/X-Learner progression already spans "cheap baseline" to "purpose-built for this setting," which is enough breadth for this stage.

## Step 3: Base learners

`HistGradientBoostingClassifier` is the primary base learner, used for every stage of every meta-learner, via `build_gradient_boosting_base_learner` in `src/models.py`. It captures nonlinearities and interactions among `recency`/`history`/the categorical dummies without hand-engineered interaction terms (left for the models to find, per `02_data_prep.ipynb`), needs no feature scaling, and is robust to `history`'s right skew (skewness 2.42, per `01_eda.ipynb`) without a log-transform. `max_depth` and `min_samples_leaf` are kept conservative relative to scikit-learn's defaults given `visit`'s moderate rarity and the per-arm training size — deep, specific leaves are exactly the overfitting shape a rare outcome invites.

A scaled `LogisticRegression` pipeline is included as a second base learner, S-Learner only (`build_logistic_regression_base_learner`), mainly to show the pipeline discipline a linear model actually requires. Unlike the boosted trees it needs numeric features on comparable scales — without scaling, `history` (range ~$30–$3,345) would dominate the regularization penalty relative to `recency` (range 1–12) — handled with a `ColumnTransformer` + `StandardScaler` fit inside the pipeline so scale parameters come fresh from each fit's own training rows. Multicollinearity is already resolved upstream (`02_data_prep.ipynb` dropped `history_segment` and one-hot encoded with `drop_first=True`), so nothing extra is needed beyond scaling. `class_weight="balanced"` is fixed, not tuned — logistic regression has no depth/leaf mechanism to attend to a minority class the way the boosted trees do.

Limited to the S-Learner rather than repeated for T-/X-Learner, to keep the meta-learner comparison as the notebook's main axis; this comparison mainly demonstrates pipeline discipline.

## Step 4: Hyperparameter search strategy

Hyperparameters are tuned lightly: once per base-learner type, on the pooled training set's `visit` outcome (ignoring treatment and arm), via `RandomizedSearchCV` scored on cross-validated average precision (PR-AUC, preferred over ROC-AUC under `visit`'s class imbalance) — `tune_base_learner_hyperparameters` in `src/models.py`.

- Tuned against outcome-prediction fit, not the uplift metric (Qini/AUUC) directly. Qini/AUUC are computed on a small, rare-outcome holdout and are themselves noisy at this project's scale; tuning against a noisy target risks fitting that noise. A component model's own predictive fit is a lower-variance proxy — necessary though not sufficient for good uplift decomposition — and the uplift metric is reserved for genuine held-out evaluation in `04_evaluation.ipynb`.
- Tuned once on pooled data, not per arm or per meta-learner stage. All arms share the same covariate distribution by randomization, so there's no reason to expect the outcome model's appropriate complexity to differ by arm; tuning each stage separately would multiply search cost for an unlikely benefit at this scale.
- `RandomizedSearchCV`, not an exhaustive grid. With four hyperparameters for the boosting model, a grid gets expensive fast, and this dataset's size and feature count (nine columns) don't justify it — 15 random combinations find a comparably good configuration for a fraction of the compute. (The logistic search has one hyperparameter with six candidate values, so `n_iter=6` there is already exhaustive.)

In [4]:
gbm_param_distributions = {
    "max_iter": [50, 100, 150],
    "max_depth": [3, 5, 7, None],
    "learning_rate": [0.03, 0.05, 0.1],
    "min_samples_leaf": [20, 50, 100],
}

tuned_gbm = tune_base_learner_hyperparameters(
    estimator=build_gradient_boosting_base_learner(),
    param_distributions=gbm_param_distributions,
    X=train_df[feature_columns],
    y=train_df["visit"],
    n_iter=15,
    cv=5,
    scoring="average_precision",
    random_state=42,
)

gbm_params = {
    key: value
    for key, value in tuned_gbm.get_params().items()
    if key in {"max_iter", "max_depth", "learning_rate", "min_samples_leaf"}
}
print("Tuned gradient-boosting config (reused across every GBM-based model below):")
gbm_params

Tuned gradient-boosting config (reused across every GBM-based model below):


{'learning_rate': 0.1, 'max_depth': 3, 'max_iter': 150, 'min_samples_leaf': 50}

In [5]:
logreg_param_distributions = {"classifier__C": np.logspace(-3, 2, 6)}

tuned_logreg_pipeline = tune_base_learner_hyperparameters(
    estimator=build_logistic_regression_base_learner(numeric_features=["recency", "history"]),
    param_distributions=logreg_param_distributions,
    X=train_df[feature_columns],
    y=train_df["visit"],
    n_iter=6,  # only 6 candidate values total, so this is effectively exhaustive
    cv=5,
    scoring="average_precision",
    random_state=42,
)

best_C = tuned_logreg_pipeline.named_steps["classifier"].C
print(f"Tuned LogisticRegression C (reused for the S-Learner logistic comparison): {best_C}")

Tuned LogisticRegression C (reused for the S-Learner logistic comparison): 100.0


## Step 5: Fitting the models

Each arm gets the same four models: S-Learner (GBM), S-Learner (Logistic), T-Learner (GBM), X-Learner (GBM) — all built from the tuned base-learner configurations above. `fit_candidate_models` below is a small notebook-local helper, not something added to `src/models.py`: it encodes *this notebook's specific* choice of which meta-learners to compare and with which base learners, which is an experimental-design decision that belongs here, not a generic reusable primitive. The building blocks it calls — the base-learner factories and `XLearner` — do live in `src/models.py`, since those are reusable across any future notebook that needs an uplift model.

In [6]:
def fit_candidate_models(train_X, train_y, train_treatment, gbm_params, best_C):
    """Fit the S-/T-/X-Learner comparison used throughout this notebook.

    Notebook-local helper (not in src/models.py): it encodes this notebook's
    specific experimental design -- which meta-learners to compare, and with
    which base learners -- rather than a generic, reusable primitive. The
    building blocks it calls (base-learner factories, XLearner) do live in
    src/models.py, since those are reusable across any future notebook that
    needs an uplift model.
    """
    models = {}

    models["S-Learner (GBM)"] = SoloModel(
        estimator=build_gradient_boosting_base_learner(**gbm_params, random_state=42)
    )
    models["S-Learner (GBM)"].fit(train_X, train_y, train_treatment)

    models["S-Learner (Logistic)"] = SoloModel(
        estimator=build_logistic_regression_base_learner(
            numeric_features=["recency", "history"], C=best_C, random_state=42
        )
    )
    models["S-Learner (Logistic)"].fit(train_X, train_y, train_treatment)

    models["T-Learner (GBM)"] = TwoModels(
        estimator_trmnt=build_gradient_boosting_base_learner(**gbm_params, random_state=42),
        estimator_ctrl=build_gradient_boosting_base_learner(**gbm_params, random_state=42),
        method="vanilla",
    )
    models["T-Learner (GBM)"].fit(train_X, train_y, train_treatment)

    models["X-Learner (GBM)"] = XLearner(
        estimator_outcome_treatment=build_gradient_boosting_base_learner(**gbm_params, random_state=42),
        estimator_outcome_control=build_gradient_boosting_base_learner(**gbm_params, random_state=42),
        estimator_effect_treatment=build_gradient_boosting_effect_regressor(**gbm_params, random_state=42),
        estimator_effect_control=build_gradient_boosting_effect_regressor(**gbm_params, random_state=42),
        propensity=None,
    )
    models["X-Learner (GBM)"].fit(train_X, train_y, train_treatment)

    return models

In [7]:
mens_splits = prepare_arm_splits(
    train_df, val_df, test_df,
    treatment_group="Mens E-Mail",
    feature_columns=feature_columns,
    outcome_col="visit",
)
X_train_mens, y_train_mens, treat_train_mens = mens_splits["train"]
X_val_mens, y_val_mens, treat_val_mens = mens_splits["val"]

mens_models = fit_candidate_models(X_train_mens, y_train_mens, treat_train_mens, gbm_params, best_C)
print("Fitted models for Mens E-Mail vs No E-Mail:", list(mens_models.keys()))

Fitted models for Mens E-Mail vs No E-Mail: ['S-Learner (GBM)', 'S-Learner (Logistic)', 'T-Learner (GBM)', 'X-Learner (GBM)']


In [8]:
womens_splits = prepare_arm_splits(
    train_df, val_df, test_df,
    treatment_group="Womens E-Mail",
    feature_columns=feature_columns,
    outcome_col="visit",
)
X_train_womens, y_train_womens, treat_train_womens = womens_splits["train"]
X_val_womens, y_val_womens, treat_val_womens = womens_splits["val"]

womens_models = fit_candidate_models(X_train_womens, y_train_womens, treat_train_womens, gbm_params, best_C)
print("Fitted models for Womens E-Mail vs No E-Mail:", list(womens_models.keys()))

Fitted models for Womens E-Mail vs No E-Mail: ['S-Learner (GBM)', 'S-Learner (Logistic)', 'T-Learner (GBM)', 'X-Learner (GBM)']


## Step 6: A first look at the results

This is a **sanity check, not the final evaluation**. The full, rigorous evaluation — Qini curves, uplift@k at business-relevant targeting budgets, and confidence intervals via bootstrap — is `04_evaluation.ipynb`'s job, run once on `test.parquet`. Here, the goal is narrower: confirm that each model's predicted uplift score actually ranks customers in a sensible way (higher predicted uplift should correspond to a bigger treated-vs-control outcome gap) before investing in that full evaluation.

`uplift_by_decile` below is a small, throwaway diagnostic defined in this notebook rather than added to `src/models.py` or a new `src/evaluation.py` — real evaluation infrastructure belongs in `04_evaluation.ipynb`, and building it prematurely here would pre-commit evaluation-metric design before that notebook is where the actual thinking about metrics should happen. `sklift.metrics.qini_auc_score` is also used directly, purely as a quick single-number cross-check alongside the decile table.

In [9]:
def uplift_by_decile(y, uplift_scores, treatment, n_bins=10):
    """Quick, throwaway diagnostic: observed uplift by predicted-uplift bin.

    Not reusable evaluation infrastructure -- that belongs in
    04_evaluation.ipynb / src/evaluation.py, deliberately not built yet so
    evaluation-metric design isn't pre-committed before that notebook's
    actual focus. This only confirms each model's ranking is plausible
    (higher predicted uplift should correspond to a bigger treated-vs-control
    outcome gap) before the dedicated evaluation stage.
    """
    frame = pd.DataFrame({
        "y": np.asarray(y),
        "uplift": np.asarray(uplift_scores),
        "treatment": np.asarray(treatment),
    })
    frame["bin"] = pd.qcut(frame["uplift"], n_bins, labels=False, duplicates="drop")

    n_actual_bins = frame["bin"].nunique()
    if n_actual_bins < n_bins:
        print(
            f"Note: requested {n_bins} bins, but only {n_actual_bins} unique "
            "quantile bins could be formed for this model's uplift scores "
            "(many tied predictions). Bin counts below are not directly "
            "comparable to a model that reached the full n_bins."
        )

    treated_rate = frame.loc[frame["treatment"] == 1].groupby("bin")["y"].mean()
    control_rate = frame.loc[frame["treatment"] == 0].groupby("bin")["y"].mean()
    n_per_bin = frame.groupby("bin").size()

    result = pd.DataFrame({"n": n_per_bin, "treated_rate": treated_rate, "control_rate": control_rate})
    result["observed_uplift"] = result["treated_rate"] - result["control_rate"]
    return result.sort_index(ascending=False)

In [10]:
mens_val_predictions = collect_uplift_predictions(mens_models, X_val_mens)

for model_name in mens_models:
    print(f"--- Mens E-Mail -- {model_name} ---")
    display(uplift_by_decile(y_val_mens, mens_val_predictions[model_name], treat_val_mens))

--- Mens E-Mail -- S-Learner (GBM) ---


,n,treated_rate,control_rate,observed_uplift
bin,,,,
9,852,0.262136,0.186364,0.075772
8,851,0.252822,0.122549,0.130273
7,854,0.181193,0.102871,0.078322
6,849,0.183908,0.125604,0.058304
5,854,0.187793,0.123832,0.063962
4,843,0.145985,0.101852,0.044134
3,860,0.171296,0.116822,0.054474
2,852,0.180095,0.093023,0.087072
1,854,0.134940,0.077449,0.057491


--- Mens E-Mail -- S-Learner (Logistic) ---


,n,treated_rate,control_rate,observed_uplift
bin,,,,
9,853,0.204128,0.095923,0.108205
8,852,0.205379,0.106095,0.099284
7,852,0.200935,0.127358,0.073576
6,852,0.231481,0.114286,0.117196
5,852,0.199017,0.125843,0.073175
4,852,0.146919,0.111628,0.035292
3,852,0.204444,0.124378,0.080066
2,852,0.140811,0.090069,0.050742
1,852,0.155211,0.089776,0.065435


--- Mens E-Mail -- T-Learner (GBM) ---


,n,treated_rate,control_rate,observed_uplift
bin,,,,
9,853,0.276102,0.182464,0.093638
8,852,0.226545,0.156627,0.069918
7,852,0.197701,0.117506,0.080195
6,852,0.174014,0.083135,0.090879
5,851,0.176334,0.107143,0.069191
4,853,0.157303,0.093137,0.064166
3,851,0.154034,0.083710,0.070324
2,853,0.127711,0.075342,0.052368
1,851,0.129771,0.098253,0.031518


--- Mens E-Mail -- X-Learner (GBM) ---


,n,treated_rate,control_rate,observed_uplift
bin,,,,
9,853,0.294840,0.186099,0.108742
8,851,0.216814,0.145363,0.071451
7,852,0.186207,0.112710,0.073497
6,851,0.153132,0.121429,0.031704
5,853,0.152620,0.096618,0.056001
4,853,0.196126,0.084091,0.112035
3,843,0.144144,0.087719,0.056425
2,861,0.172662,0.094595,0.078067
1,852,0.178922,0.103604,0.075318


In [11]:
womens_val_predictions = collect_uplift_predictions(womens_models, X_val_womens)

for model_name in womens_models:
    print(f"--- Womens E-Mail -- {model_name} ---")
    display(uplift_by_decile(y_val_womens, womens_val_predictions[model_name], treat_val_womens))

--- Womens E-Mail -- S-Learner (GBM) ---


,n,treated_rate,control_rate,observed_uplift
bin,,,,
9,852,0.187793,0.157277,0.030516
8,856,0.187204,0.117512,0.069692
7,854,0.199125,0.110831,0.088293
6,854,0.165501,0.117647,0.047854
5,851,0.176887,0.117096,0.059791
4,855,0.132701,0.094688,0.038013
3,855,0.110345,0.076190,0.034154
2,854,0.069284,0.071259,-0.001975
1,853,0.092010,0.077273,0.014737


--- Womens E-Mail -- S-Learner (Logistic) ---


,n,treated_rate,control_rate,observed_uplift
bin,,,,
9,854,0.149289,0.113426,0.035863
8,854,0.160000,0.097902,0.062098
7,854,0.134884,0.139151,-0.004267
6,854,0.145786,0.084337,0.061449
5,853,0.177677,0.142512,0.035164
4,854,0.155660,0.095349,0.060312
3,854,0.152582,0.109813,0.042769
2,854,0.165072,0.112385,0.052686
1,854,0.139211,0.096927,0.042284


--- Womens E-Mail -- T-Learner (GBM) ---


,n,treated_rate,control_rate,observed_uplift
bin,,,,
9,854,0.201856,0.170213,0.031643
8,851,0.178899,0.089157,0.089742
7,856,0.193995,0.115839,0.078156
6,848,0.168618,0.106888,0.061730
5,860,0.158508,0.081206,0.077302
4,854,0.135747,0.111650,0.024096
3,853,0.106818,0.111380,-0.004562
2,855,0.099757,0.078829,0.020928
1,853,0.113527,0.102506,0.011021


--- Womens E-Mail -- X-Learner (GBM) ---


,n,treated_rate,control_rate,observed_uplift
bin,,,,
9,847,0.180778,0.131707,0.049071
8,857,0.168182,0.081535,0.086647
7,858,0.196468,0.116049,0.080419
6,853,0.155172,0.111857,0.043316
5,854,0.187354,0.128806,0.058548
4,854,0.157895,0.124700,0.033194
3,835,0.096927,0.104369,-0.007442
2,861,0.109524,0.070295,0.039229
1,863,0.094763,0.093074,0.001689


In [12]:
qini_summary = []
for arm_name, models, X_val, y_val, treat_val in [
    ("Mens E-Mail", mens_models, X_val_mens, y_val_mens, treat_val_mens),
    ("Womens E-Mail", womens_models, X_val_womens, y_val_womens, treat_val_womens),
]:
    predictions = collect_uplift_predictions(models, X_val)
    for model_name in models:
        score = qini_auc_score(
            y_val.to_numpy(), predictions[model_name], treat_val.to_numpy()
        )
        qini_summary.append({"arm": arm_name, "model": model_name, "qini_auc": score})

qini_summary_df = pd.DataFrame(qini_summary).sort_values(["arm", "qini_auc"], ascending=[True, False])
qini_summary_df

/home/dipa/.conda/envs/tf-gpu/lib/python3.12/site-packages/sklearn/utils/deprecation.py:95: FutureWarning: Function stable_cumsum is deprecated; `sklearn.utils.extmath.stable_cumsum` is deprecated in version 1.8 and will be removed in 1.10. Use `np.cumulative_sum` with the desired dtype directly instead.
  warnings.warn(msg, category=FutureWarning)
/home/dipa/.conda/envs/tf-gpu/lib/python3.12/site-packages/sklearn/utils/deprecation.py:95: FutureWarning: Function stable_cumsum is deprecated; `sklearn.utils.extmath.stable_cumsum` is deprecated in version 1.8 and will be removed in 1.10. Use `np.cumulative_sum` with the desired dtype directly instead.
  warnings.warn(msg, category=FutureWarning)
/home/dipa/.conda/envs/tf-gpu/lib/python3.12/site-packages/sklearn/utils/deprecation.py:95: FutureWarning: Function stable_cumsum is deprecated; `sklearn.utils.extmath.stable_cumsum` is deprecated in version 1.8 and will be removed in 1.10. Use `np.cumulative_sum` with the desired dtype directly i

,arm,model,qini_auc
1,Mens E-Mail,S-Learner (Logistic),0.032715
2,Mens E-Mail,T-Learner (GBM),0.003481
0,Mens E-Mail,S-Learner (GBM),0.002928
3,Mens E-Mail,X-Learner (GBM),-0.009608
7,Womens E-Mail,X-Learner (GBM),0.058723
6,Womens E-Mail,T-Learner (GBM),0.053944
4,Womens E-Mail,S-Learner (GBM),0.045965
5,Womens E-Mail,S-Learner (Logistic),0.005831


In [13]:
px.bar(
    qini_summary_df, x="model", y="qini_auc", color="arm", barmode="group",
    title="Quick Qini AUC look across meta-learners and arms (validation set, visit target)",
).show()

## Step 7: A conversion stress test

`visit` was chosen as the primary target because `conversion` is too rare for reliable heterogeneous-effect estimation at this project's scale. This section fits one model on `conversion` to show that directly rather than just asserting it.

Scope: one arm (`Womens E-Mail` vs `No E-Mail`), one model (X-Learner, best suited to a rare/imbalanced setting among the three compared above), hand-picked conservative hyperparameters rather than a fresh search. The training data has 103 conversions among 12,832 treated rows and 68 among 12,784 control rows — a search would tune against noise, and splitting that signal further into CV folds would leave next to nothing per fold. The tuned `gbm_params` from `visit` aren't reused either: a complexity suited to a ~15%-positive outcome is too flexible for one an order of magnitude rarer, so a shallower, larger-leaved configuration is used.

Qini AUC isn't computed here: with this few conversions in validation, the number would swing on the treatment status of a handful of rows. The decile table uses quintiles instead of deciles (deciles would leave most bins with zero conversions in one arm or the other) — expect the bins to look close to noise rather than the separation the `visit` models showed.

In [14]:
womens_conversion_splits = prepare_arm_splits(
    train_df, val_df, test_df,
    treatment_group="Womens E-Mail",
    feature_columns=feature_columns,
    outcome_col="conversion",
)
X_train_conv, y_train_conv, treat_train_conv = womens_conversion_splits["train"]
X_val_conv, y_val_conv, treat_val_conv = womens_conversion_splits["val"]

n_conversions_treated = int(y_train_conv[treat_train_conv == 1].sum())
n_conversions_control = int(y_train_conv[treat_train_conv == 0].sum())
n_treated = int((treat_train_conv == 1).sum())
n_control = int((treat_train_conv == 0).sum())

print("Training data, Womens E-Mail vs No E-Mail, conversion target:")
print(f"  Treated arm: {n_conversions_treated} conversions out of {n_treated} rows")
print(f"  Control arm: {n_conversions_control} conversions out of {n_control} rows")

Training data, Womens E-Mail vs No E-Mail, conversion target:
  Treated arm: 103 conversions out of 12832 rows
  Control arm: 68 conversions out of 12784 rows


In [15]:
conservative_gbm_params = {
    "max_iter": 50,
    "max_depth": 3,
    "learning_rate": 0.05,
    "min_samples_leaf": 100,
}

conversion_x_learner = XLearner(
    estimator_outcome_treatment=build_gradient_boosting_base_learner(**conservative_gbm_params, random_state=42),
    estimator_outcome_control=build_gradient_boosting_base_learner(**conservative_gbm_params, random_state=42),
    estimator_effect_treatment=build_gradient_boosting_effect_regressor(**conservative_gbm_params, random_state=42),
    estimator_effect_control=build_gradient_boosting_effect_regressor(**conservative_gbm_params, random_state=42),
    propensity=None,
)
conversion_x_learner.fit(X_train_conv, y_train_conv, treat_train_conv)

conversion_val_uplift = conversion_x_learner.predict(X_val_conv)
uplift_by_decile(y_val_conv, conversion_val_uplift, treat_val_conv, n_bins=5)

,n,treated_rate,control_rate,observed_uplift
bin,,,,
4,1689,0.018476,0.007290,0.011185
3,1727,0.012821,0.002301,0.010519
2,1706,0.010022,0.003713,0.006309
1,1706,0.009975,0.002212,0.007763
0,1711,0.004684,0.008168,-0.003484


## Step 8: Two more baselines — Decision Tree and Random Forest

Included because they were requested, not because either is expected to suit this task. Neither has an uplift-specific mechanism:

- A single decision tree is high-variance (small sample changes reshape the whole split structure) and, as an S-Learner base learner, optimizes outcome-prediction purity, not treatment-effect separation. `treatment` is just another feature to it.
- A random forest averages many such trees, which reduces variance but not the underlying issue. A genuine uplift forest would split on a treatment-effect divergence criterion; a `RandomForestClassifier` inside an S-Learner doesn't — it's the same outcome-prediction setup as the tree, just averaged.

Neither is expected to beat the tuned `HistGradientBoostingClassifier` results from Step 6. Both stay S-Learner-only, same scope as the logistic-regression comparison in Step 3, so the meta-learner comparison stays the notebook's main axis.

`build_decision_tree_base_learner` and `build_random_forest_base_learner` (new in `src/models.py`) follow this project's base-learner conventions: `random_state`, `class_weight="balanced"` (same imbalance fix as logistic regression), and conservative depth/leaf defaults for this project's per-arm training size (~25k rows) and `visit`'s ~15% positive rate.

Both are added directly into the existing `mens_models` / `womens_models` dicts from Step 5; `fit_candidate_models` itself isn't touched.

In [16]:
mens_models["S-Learner (Decision Tree)"] = SoloModel(
    estimator=build_decision_tree_base_learner(random_state=42)
)
mens_models["S-Learner (Decision Tree)"].fit(X_train_mens, y_train_mens, treat_train_mens)

mens_models["S-Learner (Random Forest)"] = SoloModel(
    estimator=build_random_forest_base_learner(random_state=42)
)
mens_models["S-Learner (Random Forest)"].fit(X_train_mens, y_train_mens, treat_train_mens)

print("Mens E-Mail models now:", list(mens_models.keys()))


Mens E-Mail models now: ['S-Learner (GBM)', 'S-Learner (Logistic)', 'T-Learner (GBM)', 'X-Learner (GBM)', 'S-Learner (Decision Tree)', 'S-Learner (Random Forest)']


In [17]:
womens_models["S-Learner (Decision Tree)"] = SoloModel(
    estimator=build_decision_tree_base_learner(random_state=42)
)
womens_models["S-Learner (Decision Tree)"].fit(X_train_womens, y_train_womens, treat_train_womens)

womens_models["S-Learner (Random Forest)"] = SoloModel(
    estimator=build_random_forest_base_learner(random_state=42)
)
womens_models["S-Learner (Random Forest)"].fit(X_train_womens, y_train_womens, treat_train_womens)

print("Womens E-Mail models now:", list(womens_models.keys()))


Womens E-Mail models now: ['S-Learner (GBM)', 'S-Learner (Logistic)', 'T-Learner (GBM)', 'X-Learner (GBM)', 'S-Learner (Decision Tree)', 'S-Learner (Random Forest)']


## Step 9: A neural, uplift-native model — TARNet

TARNet (Treatment-Agnostic Representation Network, Shalit, Johansson & Sontag, ICML 2017). A shared feed-forward trunk (2-3 FC layers, ReLU + dropout) learns a representation of the covariates; two independent linear heads sit on top of that same representation, one predicting `P(visit=1 | control)`, one `P(visit=1 | treatment)`. Predicted uplift is `treatment_head(phi(x)) - control_head(phi(x))`. Implemented as `TARNetUpliftModel` in `src/models.py`.

**Plain TARNet, not CFRNet or DragonNet.** Both extensions correct for treatment assignment correlating with covariates in observational data — CFRNet with an IPM penalty pushing treated/control representations to look similar, DragonNet with a propensity head that regularizes the representation. This is a confirmed-randomized experiment (`01_eda.ipynb`'s SMD balance check passed on every covariate/arm combination; `XLearner` already uses the known randomization probability rather than an estimated propensity, see `decisions_log.md`), so there's no covariate mismatch for CFRNet to fix and no unknown propensity for DragonNet to estimate. Plain TARNet already gives the part that matters regardless of randomization: two arm-specific output heads on a shared, jointly-learned representation, the middle ground between S-Learner (one global model) and T-Learner (two fully separate ones).

Training uses factual loss only: each row's `BCEWithLogitsLoss` is computed against the head matching its actually-observed arm, never the counterfactual one. `pos_weight` is computed per arm from that arm's own training-split class balance — the neural analogue of `class_weight="balanced"` used elsewhere in this project.

`.fit(X, y, treatment)` / `.predict(X)` match `XLearner`'s interface, so `TARNetUpliftModel` drops into `collect_uplift_predictions`, `qini_auc_score`, the decile diagnostic, and Step 11's ensembling with no special-casing. `.fit` also takes optional `X_val`/`y_val`/`treatment_val` for early stopping only.

Feature scaling: same `ColumnTransformer` + `StandardScaler` pattern as the logistic base learner (`recency`, `history` scaled, dummies passed through), fit only on the data passed to `.fit()`.

Hyperparameters are light, conservative defaults rather than a search: hidden sizes `[64, 32]`, dropout 0.2, Adam with small weight decay, lr `1e-3`, batch size 256, up to 150 epochs with early stopping (patience 12) on validation loss, using the existing `X_val_mens`/`X_val_womens` splits. `torch`, `numpy`, and Python's `random` seeds are set inside `.fit` for reproducibility.

One `TARNetUpliftModel` per arm, fit on `visit`, added to `mens_models`/`womens_models`.

`build_mlp_base_learner` (new in `src/models.py`) wraps a standard `MLPClassifier` in the same scaling pipeline and is fit as one more S-Learner comparison, for contrast: it's a generic classifier that sees `treatment` as an ordinary feature and inherits the S-Learner failure mode from Step 2, where TARNet is structurally incapable of ignoring treatment — separate heads by construction.

In [18]:
from sklearn.model_selection import train_test_split

# Early-stopping split carved out of TRAIN only. X_val_mens must stay
# completely unseen by every model until Step 10/11 -- previously TARNet
# was the only model whose .fit() saw X_val_mens/y_val_mens (via early
# stopping), giving it an undisclosed advantage when later ranked against
# every other model on that same "held-out" validation set.
X_fit_mens, X_es_mens, y_fit_mens, y_es_mens, treat_fit_mens, treat_es_mens = train_test_split(
    X_train_mens, y_train_mens, treat_train_mens,
    test_size=0.15, stratify=treat_train_mens, random_state=42,
)

mens_tarnet = TARNetUpliftModel(
    numeric_features=["recency", "history"],
    hidden_sizes=(64, 32),
    dropout=0.2,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=256,
    max_epochs=150,
    patience=12,
    random_state=42,
)
mens_tarnet.fit(
    X_fit_mens, y_fit_mens, treat_fit_mens,
    X_val=X_es_mens, y_val=y_es_mens, treatment_val=treat_es_mens,
)
mens_models["TARNet"] = mens_tarnet

n_epochs_run = len(mens_tarnet.history_["train_loss"])
print(f"Mens E-Mail -- TARNet ran {n_epochs_run} epoch(s), "
      f"best (lowest validation loss) epoch: {mens_tarnet.history_['best_epoch']}")

Mens E-Mail -- TARNet ran 34 epoch(s), best (lowest validation loss) epoch: 21


In [19]:
X_fit_womens, X_es_womens, y_fit_womens, y_es_womens, treat_fit_womens, treat_es_womens = train_test_split(
    X_train_womens, y_train_womens, treat_train_womens,
    test_size=0.15, stratify=treat_train_womens, random_state=42,
)

womens_tarnet = TARNetUpliftModel(
    numeric_features=["recency", "history"],
    hidden_sizes=(64, 32),
    dropout=0.2,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=256,
    max_epochs=150,
    patience=12,
    random_state=42,
)
womens_tarnet.fit(
    X_fit_womens, y_fit_womens, treat_fit_womens,
    X_val=X_es_womens, y_val=y_es_womens, treatment_val=treat_es_womens,
)
womens_models["TARNet"] = womens_tarnet

n_epochs_run = len(womens_tarnet.history_["train_loss"])
print(f"Womens E-Mail -- TARNet ran {n_epochs_run} epoch(s), "
      f"best (lowest validation loss) epoch: {womens_tarnet.history_['best_epoch']}")

Womens E-Mail -- TARNet ran 35 epoch(s), best (lowest validation loss) epoch: 22


In [20]:
mens_models["S-Learner (MLP)"] = SoloModel(
    estimator=build_mlp_base_learner(numeric_features=["recency", "history"], random_state=42)
)
mens_models["S-Learner (MLP)"].fit(X_train_mens, y_train_mens, treat_train_mens)

womens_models["S-Learner (MLP)"] = SoloModel(
    estimator=build_mlp_base_learner(numeric_features=["recency", "history"], random_state=42)
)
womens_models["S-Learner (MLP)"].fit(X_train_womens, y_train_womens, treat_train_womens)

print("Mens E-Mail models now:", list(mens_models.keys()))
print("Womens E-Mail models now:", list(womens_models.keys()))


Mens E-Mail models now: ['S-Learner (GBM)', 'S-Learner (Logistic)', 'T-Learner (GBM)', 'X-Learner (GBM)', 'S-Learner (Decision Tree)', 'S-Learner (Random Forest)', 'TARNet', 'S-Learner (MLP)']
Womens E-Mail models now: ['S-Learner (GBM)', 'S-Learner (Logistic)', 'T-Learner (GBM)', 'X-Learner (GBM)', 'S-Learner (Decision Tree)', 'S-Learner (Random Forest)', 'TARNet', 'S-Learner (MLP)']


### Training diagnostics: loss curves and per-epoch metrics

Before trusting a freshly-trained neural model at all: did it converge, and did early stopping fire at a sensible point.

Loss curves: for each arm's TARNet fit, training vs. validation loss per epoch, early-stopping epoch marked — two charts (Mens, Womens), matching the notebook's existing per-arm cell pattern (Steps 5 and 6 both fit/inspect each arm separately).

Validation average precision, per head: tracked and plotted per epoch, factual predictions against validation rows in that head's own arm. Average precision over ROC-AUC for the same imbalance reason as Step 4.

Not plotted: a per-epoch Qini/uplift curve — at this validation size and `visit`'s prevalence it would be dominated by sampling noise, same reasoning as skipping Qini AUC on the `conversion` stress test in Step 7, applied here at epoch granularity. Loss and per-head average precision converging is a lower-variance proxy for "did this train sensibly"; the actual uplift comparison happens in Step 10 (and rigorously in `04_evaluation.ipynb`).

Also skipped: the already-tuned `HistGradientBoostingClassifier`'s training/validation deviance across boosting iterations — would need refitting with early-stopping/validation tracking enabled, for a diagnostic on a model Step 6 already showed ranking sensibly.

In [21]:
def _tarnet_loss_curve_df(history):
    """Reshape a fitted TARNetUpliftModel's history_ into a long-format
    frame for plotting train vs. validation loss per epoch."""
    n_epochs = len(history["train_loss"])
    return pd.DataFrame({
        "epoch": range(n_epochs),
        "train_loss": history["train_loss"],
        "val_loss": history["val_loss"],
    })


def _tarnet_ap_curve_df(history):
    """Reshape a fitted TARNetUpliftModel's history_ into a long-format
    frame of per-epoch validation average precision, one column per head."""
    n_epochs = len(history["val_avg_precision_control"])
    return pd.DataFrame({
        "epoch": range(n_epochs),
        "control_head_ap": history["val_avg_precision_control"],
        "treatment_head_ap": history["val_avg_precision_treatment"],
    })


In [22]:
mens_loss_df = _tarnet_loss_curve_df(mens_tarnet.history_)

fig = px.line(
    mens_loss_df, x="epoch", y=["train_loss", "val_loss"],
    title="TARNet training curve -- Mens E-Mail (dashed line: early-stopping epoch)",
)
fig.add_vline(x=mens_tarnet.history_["best_epoch"], line_dash="dash", line_color="gray")
fig.show()


In [23]:
womens_loss_df = _tarnet_loss_curve_df(womens_tarnet.history_)

fig = px.line(
    womens_loss_df, x="epoch", y=["train_loss", "val_loss"],
    title="TARNet training curve -- Womens E-Mail (dashed line: early-stopping epoch)",
)
fig.add_vline(x=womens_tarnet.history_["best_epoch"], line_dash="dash", line_color="gray")
fig.show()


In [24]:
mens_ap_df = _tarnet_ap_curve_df(mens_tarnet.history_)

px.line(
    mens_ap_df, x="epoch", y=["control_head_ap", "treatment_head_ap"],
    title="TARNet validation average precision by head, per epoch -- Mens E-Mail",
    labels={"value": "average precision", "variable": "head"},
).show()


In [25]:
womens_ap_df = _tarnet_ap_curve_df(womens_tarnet.history_)

px.line(
    womens_ap_df, x="epoch", y=["control_head_ap", "treatment_head_ap"],
    title="TARNet validation average precision by head, per epoch -- Womens E-Mail",
    labels={"value": "average precision", "variable": "head"},
).show()


## Step 10: Consolidated evaluation metrics across every model

Step 6 took a first, narrow look (decile tables plus a quick Qini AUC number) at the original four models. This builds one comparison table over the full `mens_models`/`womens_models` dicts — the original four plus Decision Tree, Random Forest, MLP, and TARNet — using the same iteration pattern as Step 6. Step 6's own cells aren't modified.

For each arm and model: Qini AUC (`sklift.metrics.qini_auc_score`) plus uplift at the top 10/20/30% of customers by predicted uplift, via `compute_uplift_at_k` (new in `src/models.py`).

`compute_uplift_at_k` is a lightweight, provisional metric for this notebook's own informal comparison — its docstring says so. The authoritative implementation (bootstrap confidence intervals, business-relevant budget framing, proper Qini-curve machinery) belongs in `src/evaluation.py`, built fresh for `04_evaluation.ipynb` — the same boundary Step 6 already draws for `uplift_by_decile`.

This ranking is informal and validation-based: a provisional best-candidate model per arm to carry forward, nothing more. Final model selection with confidence intervals happens in `04_evaluation.ipynb`, on the still-untouched `test.parquet`.

In [26]:
mens_val_predictions_all = collect_uplift_predictions(mens_models, X_val_mens)
womens_val_predictions_all = collect_uplift_predictions(womens_models, X_val_womens)

full_comparison_rows = []
for arm_name, models, y_val, treat_val, predictions in [
    ("Mens E-Mail", mens_models, y_val_mens, treat_val_mens, mens_val_predictions_all),
    ("Womens E-Mail", womens_models, y_val_womens, treat_val_womens, womens_val_predictions_all),
]:
    for model_name in models:
        uplift_scores = predictions[model_name]
        full_comparison_rows.append({
            "arm": arm_name,
            "model": model_name,
            "qini_auc": qini_auc_score(y_val.to_numpy(), uplift_scores, treat_val.to_numpy()),
            "uplift_at_10pct": compute_uplift_at_k(y_val, uplift_scores, treat_val, k=0.10),
            "uplift_at_20pct": compute_uplift_at_k(y_val, uplift_scores, treat_val, k=0.20),
            "uplift_at_30pct": compute_uplift_at_k(y_val, uplift_scores, treat_val, k=0.30),
        })

full_comparison_df = (
    pd.DataFrame(full_comparison_rows)
    .sort_values(["arm", "qini_auc"], ascending=[True, False])
    .reset_index(drop=True)
)
full_comparison_df


/home/dipa/.conda/envs/tf-gpu/lib/python3.12/site-packages/sklearn/utils/deprecation.py:95: FutureWarning: Function stable_cumsum is deprecated; `sklearn.utils.extmath.stable_cumsum` is deprecated in version 1.8 and will be removed in 1.10. Use `np.cumulative_sum` with the desired dtype directly instead.
  warnings.warn(msg, category=FutureWarning)
/home/dipa/.conda/envs/tf-gpu/lib/python3.12/site-packages/sklearn/utils/deprecation.py:95: FutureWarning: Function stable_cumsum is deprecated; `sklearn.utils.extmath.stable_cumsum` is deprecated in version 1.8 and will be removed in 1.10. Use `np.cumulative_sum` with the desired dtype directly instead.
  warnings.warn(msg, category=FutureWarning)
/home/dipa/.conda/envs/tf-gpu/lib/python3.12/site-packages/sklearn/utils/deprecation.py:95: FutureWarning: Function stable_cumsum is deprecated; `sklearn.utils.extmath.stable_cumsum` is deprecated in version 1.8 and will be removed in 1.10. Use `np.cumulative_sum` with the desired dtype directly i

,arm,model,qini_auc,uplift_at_10pct,uplift_at_20pct,uplift_at_30pct
0,Mens E-Mail,S-Learner (Logistic),0.032715,0.108205,0.103571,0.093643
1,Mens E-Mail,S-Learner (MLP),0.010224,0.084515,0.088032,0.075423
2,Mens E-Mail,T-Learner (GBM),0.003481,0.093638,0.081499,0.080995
3,Mens E-Mail,S-Learner (GBM),0.002928,0.073927,0.101532,0.093373
4,Mens E-Mail,S-Learner (Decision Tree),-0.005993,0.095091,0.072231,0.068332
5,Mens E-Mail,X-Learner (GBM),-0.009608,0.108742,0.087787,0.081918
6,Mens E-Mail,TARNet,-0.018387,0.052028,0.055174,0.062030
7,Mens E-Mail,S-Learner (Random Forest),-0.025893,0.044926,0.054943,0.053149
8,Womens E-Mail,TARNet,0.064605,0.102611,0.078166,0.078268
9,Womens E-Mail,X-Learner (GBM),0.058723,0.048388,0.067583,0.072377


In [27]:
px.bar(
    full_comparison_df, x="model", y="qini_auc", color="arm", barmode="group",
    title="Full model comparison -- Qini AUC across every fitted model (validation set, visit target)",
).show()


## Step 11: Ensemble of top-performing models

"Ensemble" needs a definition here since it's ambiguous for uplift models: rank every fitted model per arm by validation Qini AUC (Step 10's table), take the top-K (default `K=3`, exposed as a parameter), and average their predicted uplift scores directly — uniform by default, with an optional Qini-AUC-proportional weighting shown afterward as a documented variant. Standard variance-reduction practice for CATE/uplift estimation, the same logic as a random forest averaging trees.

Deliberately not `sklearn.ensemble.VotingClassifier`/`StackingClassifier` — both target a single outcome label and combine classifiers predicting that label, with no concept of a continuous per-individual treatment-effect score to combine. Averaging the already-computed uplift scores directly is the natural analogue here, and it's the only option that treats a GBM-based S-Learner, a hand-rolled `XLearner`, and a TARNet identically — all three already reduce to a per-row uplift score.

Implemented as `ensemble_uplift_predictions` in `src/models.py`. `"Ensemble (Top-3)"` is added as one more row per arm into Step 10's comparison table and bar chart, directly comparable to every individual model.

Caveat on the Qini-AUC-proportional weighting variant: raw Qini AUC can be negative for a poorly-ranking model, and using a possibly-negative number as an averaging weight only behaves sensibly when every weighted model has a positive score — true for the top-3 here (that's why they're top-3), but this isn't a robustified general-purpose weighting scheme, just a documented illustration.

Considered and rejected: a mixed pipeline chaining a differentiable model (TARNet) with a non-differentiable one (a decision tree) — no established evaluation methodology, unclear gradient/fitting-order semantics across that boundary, and no benefit over the uplift-score-averaging ensemble above, which already gets model-class diversity without needing any model to know about another during fitting.

In [28]:
TOP_K = 3

ensemble_rows = []
for arm_name, predictions, y_val, treat_val in [
    ("Mens E-Mail", mens_val_predictions_all, y_val_mens, treat_val_mens),
    ("Womens E-Mail", womens_val_predictions_all, y_val_womens, treat_val_womens),
]:
    arm_ranking = full_comparison_df.loc[full_comparison_df["arm"] == arm_name].sort_values(
        "qini_auc", ascending=False
    )
    top_k_models = arm_ranking["model"].head(TOP_K).tolist()
    print(f"{arm_name}: top-{TOP_K} models by validation Qini AUC -> {top_k_models}")

    ensemble_scores = ensemble_uplift_predictions(predictions, top_k_models, weights=None)

    ensemble_rows.append({
        "arm": arm_name,
        "model": f"Ensemble (Top-{TOP_K})",
        "qini_auc": qini_auc_score(y_val.to_numpy(), ensemble_scores, treat_val.to_numpy()),
        "uplift_at_10pct": compute_uplift_at_k(y_val, ensemble_scores, treat_val, k=0.10),
        "uplift_at_20pct": compute_uplift_at_k(y_val, ensemble_scores, treat_val, k=0.20),
        "uplift_at_30pct": compute_uplift_at_k(y_val, ensemble_scores, treat_val, k=0.30),
    })

final_comparison_df = (
    pd.concat([full_comparison_df, pd.DataFrame(ensemble_rows)], ignore_index=True)
    .sort_values(["arm", "qini_auc"], ascending=[True, False])
    .reset_index(drop=True)
)
final_comparison_df


Mens E-Mail: top-3 models by validation Qini AUC -> ['S-Learner (Logistic)', 'S-Learner (MLP)', 'T-Learner (GBM)']
Womens E-Mail: top-3 models by validation Qini AUC -> ['TARNet', 'X-Learner (GBM)', 'S-Learner (Random Forest)']


/home/dipa/.conda/envs/tf-gpu/lib/python3.12/site-packages/sklearn/utils/deprecation.py:95: FutureWarning: Function stable_cumsum is deprecated; `sklearn.utils.extmath.stable_cumsum` is deprecated in version 1.8 and will be removed in 1.10. Use `np.cumulative_sum` with the desired dtype directly instead.
  warnings.warn(msg, category=FutureWarning)
/home/dipa/.conda/envs/tf-gpu/lib/python3.12/site-packages/sklearn/utils/deprecation.py:95: FutureWarning: Function stable_cumsum is deprecated; `sklearn.utils.extmath.stable_cumsum` is deprecated in version 1.8 and will be removed in 1.10. Use `np.cumulative_sum` with the desired dtype directly instead.
  warnings.warn(msg, category=FutureWarning)


,arm,model,qini_auc,uplift_at_10pct,uplift_at_20pct,uplift_at_30pct
0,Mens E-Mail,S-Learner (Logistic),0.032715,0.108205,0.103571,0.093643
1,Mens E-Mail,Ensemble (Top-3),0.015522,0.086489,0.077857,0.083793
2,Mens E-Mail,S-Learner (MLP),0.010224,0.084515,0.088032,0.075423
3,Mens E-Mail,T-Learner (GBM),0.003481,0.093638,0.081499,0.080995
4,Mens E-Mail,S-Learner (GBM),0.002928,0.073927,0.101532,0.093373
5,Mens E-Mail,S-Learner (Decision Tree),-0.005993,0.095091,0.072231,0.068332
6,Mens E-Mail,X-Learner (GBM),-0.009608,0.108742,0.087787,0.081918
7,Mens E-Mail,TARNet,-0.018387,0.052028,0.055174,0.062030
8,Mens E-Mail,S-Learner (Random Forest),-0.025893,0.044926,0.054943,0.053149
9,Womens E-Mail,TARNet,0.064605,0.102611,0.078166,0.078268


In [29]:
px.bar(
    final_comparison_df, x="model", y="qini_auc", color="arm", barmode="group",
    title="Model comparison including the Top-3 ensemble -- Qini AUC (validation set, visit target)",
).show()


In [30]:
# Documented variant, illustration only (not used in final_comparison_df
# above, where uniform averaging is the definition in effect): weight each
# ensembled model's contribution by its own validation Qini AUC instead of
# averaging uniformly. Shown for Mens E-Mail only.
mens_top_k_ranking = (
    full_comparison_df.loc[full_comparison_df["arm"] == "Mens E-Mail"]
    .sort_values("qini_auc", ascending=False)
    .head(TOP_K)
)
weighted_ensemble_scores = ensemble_uplift_predictions(
    mens_val_predictions_all,
    mens_top_k_ranking["model"].tolist(),
    weights=mens_top_k_ranking["qini_auc"].tolist(),
)
weighted_score = qini_auc_score(
    y_val_mens.to_numpy(), weighted_ensemble_scores, treat_val_mens.to_numpy()
)

uniform_score = final_comparison_df.loc[
    (final_comparison_df["arm"] == "Mens E-Mail")
    & (final_comparison_df["model"] == f"Ensemble (Top-{TOP_K})"),
    "qini_auc",
].iloc[0]

print(f"Mens E-Mail -- uniform Top-{TOP_K} ensemble Qini AUC:            {uniform_score:.4f}")
print(f"Mens E-Mail -- Qini-AUC-weighted Top-{TOP_K} ensemble Qini AUC:  {weighted_score:.4f}")


Mens E-Mail -- uniform Top-3 ensemble Qini AUC:            0.0155
Mens E-Mail -- Qini-AUC-weighted Top-3 ensemble Qini AUC:  0.0347


/home/dipa/.conda/envs/tf-gpu/lib/python3.12/site-packages/sklearn/utils/deprecation.py:95: FutureWarning: Function stable_cumsum is deprecated; `sklearn.utils.extmath.stable_cumsum` is deprecated in version 1.8 and will be removed in 1.10. Use `np.cumulative_sum` with the desired dtype directly instead.
  warnings.warn(msg, category=FutureWarning)


## Summary

- Two parallel binary uplift analyses -- Mens E-Mail vs No E-Mail, Womens E-Mail vs No E-Mail -- each now comparing **eight** models on the `visit` target: the original S-/T-/X-Learner (GBM), S-Learner (Logistic) from Steps 1-7, plus S-Learner (Decision Tree), S-Learner (Random Forest), S-Learner (MLP), and TARNet added in Steps 8-9.
- `src/models.py` gained six new pieces: `build_decision_tree_base_learner`, `build_random_forest_base_learner`, and `build_mlp_base_learner` (base-learner factories following this project's existing conventions -- `random_state`, `class_weight="balanced"`, conservative defaults for this data scale); `TARNetUpliftModel`, a from-scratch PyTorch implementation of Shalit, Johansson & Sontag's TARNet architecture, deliberately kept plain rather than extended to CFRNet or DragonNet because this is a confirmed-randomized experiment with nothing for either extension to correct; `compute_uplift_at_k`, an explicitly provisional uplift@k metric for this notebook's own informal comparisons only; and `ensemble_uplift_predictions`, a uniform-or-weighted uplift-score averager.
- Two tree baselines (Decision Tree, Random Forest) and one generic neural baseline (MLP) were included because they were requested, with the honest expectation set in advance that none would beat the tuned GBM-based models -- none of the three has any uplift-specific mechanism, unlike TARNet's shared-trunk/two-head architecture.
- Step 9 tracked and plotted TARNet's training and validation loss per epoch (with the early-stopping epoch marked) and per-head validation average precision, and deliberately did *not* plot a per-epoch Qini/uplift curve, for the same rare-outcome-noise reasoning already used to justify skipping Qini AUC on the `conversion` stress test in Step 7.
- Step 10 built one consolidated, informal comparison table and chart (Qini AUC, uplift@10/20/30%) across all eight models per arm, extending Step 6's original "first look" without modifying it. Step 11 added a ninth entry per arm, "Ensemble (Top-3)" -- the uniform average of each arm's top-3 models by validation Qini AUC -- directly comparable to every individual model in the same table, plus a documented (not adopted as default) Qini-AUC-weighted variant.
- Every ranking in Steps 8-11, like Step 6's original "first look," is informal and validation-based. Nothing in this extension touched `test.parquet`.

Next: `04_evaluation.ipynb` evaluates this full nine-way comparison (per arm) rigorously on the held-out test set -- Qini curves, uplift@k at business-relevant budgets, and bootstrap confidence intervals, with a from-scratch `src/evaluation.py` -- before `05_heterogeneity_shap.ipynb` digs into *which* covariates drive the estimated heterogeneity.
